<a href="https://colab.research.google.com/github/divyadharshini-1306/ShiftSafeAI/blob/main/ShiftSafeAI_data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/ShiftSafe_AI/data/raw', exist_ok=True)
os.makedirs('/content/drive/MyDrive/ShiftSafe_AI/data/processed', exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/ShiftSafe_AI/data/city_hour.csv')
# Filter to Bengaluru only
blr = df[df['City'] == 'Bengaluru'].copy()
blr = blr.sort_values('Datetime').reset_index(drop=True)

print(f"Total rows: {len(blr)}")
print(f"Columns: {blr.columns.tolist()}")
blr.head()

Total rows: 48192
Columns: ['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'AQI', 'AQI_Bucket']


,City,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Bengaluru,2015-01-01 01:00:00,NaN,NaN,2.04,12.40,7.58,19.10,0.19,4.05,12.41,0.56,3.43,NaN,NaN,NaN
1,Bengaluru,2015-01-01 02:00:00,NaN,NaN,2.20,12.11,7.54,17.81,0.21,4.23,12.13,NaN,4.54,NaN,NaN,NaN
2,Bengaluru,2015-01-01 03:00:00,NaN,NaN,1.66,10.82,6.52,17.42,0.30,4.28,13.13,NaN,4.20,NaN,NaN,NaN
3,Bengaluru,2015-01-01 04:00:00,NaN,NaN,1.92,10.32,6.46,16.86,0.13,4.14,11.82,NaN,4.63,NaN,NaN,NaN
4,Bengaluru,2015-01-01 05:00:00,NaN,NaN,1.94,9.82,6.22,16.35,0.15,4.26,10.31,NaN,3.91,NaN,NaN,NaN


In [ ]:
# Date range and missing value summary
blr['Datetime'] = pd.to_datetime(blr['Datetime'])
print(f"Date range: {blr['Datetime'].min()} → {blr['Datetime'].max()}")
print(f"Total hours: {len(blr)}")
print(f"Expected hours (5.5 yrs): {int(5.5 * 365 * 24)}")
print(f"\nMissing % per column:")
print((blr.isnull().mean() * 100).round(1).sort_values(ascending=False))

Date range: 2015-01-01 01:00:00 → 2020-07-01 00:00:00
Total hours: 48192
Expected hours (5.5 yrs): 48180

Missing % per column:
Xylene        100.0
PM10           22.8
Benzene        20.9
NH3            15.8
Toluene        14.9
O3             10.9
PM2.5           9.6
CO              9.6
AQI_Bucket      5.6
AQI             5.6
NOx             4.3
SO2             1.5
NO              1.3
NO2             1.3
Datetime        0.0
City            0.0
dtype: float64


In [ ]:
# We are dropping 5 columns that are either completely empty, chemically irrelevant to outdoor AQI prediction, or redundant.
# - Xylene: 100% missing for Bengaluru, nothing to salvage
# - Benzene: 20.9% missing + it's an industrial solvent, not an AQI pollutant
# - Toluene: 14.9% missing + same reason as Benzene
# - NOx: redundant — it is just NO + NO2 added together, which we already have
# - AQI_Bucket: text label like "Moderate" or "Poor" — models need numbers not text

cols_to_drop = ['Xylene', 'Benzene', 'Toluene', 'NOx', 'AQI_Bucket']
blr = blr.drop(columns=cols_to_drop)

# Confirm the drop worked — should now show 11 columns
print(f"Columns remaining: {blr.columns.tolist()}")
print(f"Shape after dropping: {blr.shape}")

Columns remaining: ['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NH3', 'CO', 'SO2', 'O3', 'AQI']
Shape after dropping: (48192, 11)


In [ ]:
from sklearn.impute import KNNImputer

# KNN Imputation fills missing values intelligently.
# For each missing cell, it finds the 5 most similar rows
# based on all other columns and fills in their average.
# Example: if PM2.5 is missing but NO2, CO, SO2 are present,
# it finds 5 rows with similar NO2/CO/SO2 values and uses
# their PM2.5 average as the fill — much smarter than column mean.

# We only impute numeric columns — Datetime and City are not numeric
numeric_cols = ['PM2.5', 'PM10', 'NO', 'NO2', 'NH3', 'CO', 'SO2', 'O3', 'AQI']

# n_neighbors=5 means it looks at 5 most similar rows to fill each gap
imputer = KNNImputer(n_neighbors=5)

# fit_transform learns the patterns and fills the gaps in one step
# We replace only the numeric columns, leaving Datetime and City untouched
blr[numeric_cols] = imputer.fit_transform(blr[numeric_cols])

# Verify — every column should now show 0 missing values
print("Missing values after imputation:")
print(blr[numeric_cols].isnull().sum())

# Also confirm no column accidentally became all zeros or negatives
print("\nQuick stats after imputation:")
print(blr[numeric_cols].describe().round(2))

Missing values after imputation:
PM2.5    0
PM10     0
NO       0
NO2      0
NH3      0
CO       0
SO2      0
O3       0
AQI      0
dtype: int64

Quick stats after imputation:
          PM2.5      PM10        NO       NO2       NH3        CO       SO2  \
count  48192.00  48192.00  48192.00  48192.00  48192.00  48192.00  48192.00   
mean      36.15     77.22      9.46     28.01     22.18      1.73      5.49   
std       33.14     45.83     12.44     19.16     15.57      3.77      6.52   
min        0.01      0.01      0.03      0.06      0.02      0.00      0.01   
25%       19.36     47.41      3.44     15.87     13.76      0.58      3.29   
50%       29.80     68.73      5.91     24.36     19.53      0.83      4.77   
75%       45.00     97.21     10.87     34.39     26.04      1.16      6.30   
max      999.99    999.99    287.04    362.12    485.82     49.96    198.30   

             O3       AQI  
count  48192.00  48192.00  
mean      32.53     94.30  
std       25.03     48.32  


In [ ]:
# IQR stands for Interquartile Range — the spread of the middle 50% of values.
# Real sensor data sometimes has glitch readings — a PM2.5 of 9999
# or a negative CO reading. These are not real pollution events.
# They would confuse the model badly if left in.
#
# We clip values to a reasonable ceiling/floor using IQR:
# - Q1 = 25th percentile (lower boundary of middle 50%)
# - Q3 = 75th percentile (upper boundary of middle 50%)
# - IQR = Q3 - Q1
# - Anything below Q1 - 3*IQR or above Q3 + 3*IQR gets clipped
#
# We use factor=3.0 (not the common 1.5) because AQI data has real
# pollution spikes during events like Diwali or dust storms.
# A tight factor=1.5 would wrongly clip those real events.
# factor=3.0 only removes genuine sensor errors.

def clip_iqr(series, factor=3.0):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    # clip() caps all values between lower and upper — no rows are deleted
    return series.clip(lower=lower, upper=upper)

# Apply IQR clipping to every numeric pollutant column
for col in numeric_cols:
    blr[col] = clip_iqr(blr[col])

# Check the result — look at min and max values
# No column should have extreme outliers anymore
print("Stats after IQR clipping:")
print(blr[numeric_cols].describe().round(2))

Stats after IQR clipping:
          PM2.5      PM10        NO       NO2       NH3        CO       SO2  \
count  48192.00  48192.00  48192.00  48192.00  48192.00  48192.00  48192.00   
mean      34.94     76.71      8.62     27.58     21.57      1.02      5.07   
std       21.78     42.63      7.60     16.80     11.71      0.72      2.72   
min        0.01      0.01      0.03      0.06      0.02      0.00      0.01   
25%       19.36     47.41      3.44     15.87     13.76      0.58      3.29   
50%       29.80     68.73      5.91     24.36     19.53      0.83      4.77   
75%       45.00     97.21     10.87     34.39     26.04      1.16      6.30   
max      121.92    246.61     33.16     89.95     62.88      2.90     15.33   

             O3       AQI  
count  48192.00  48192.00  
mean      32.44     93.10  
std       24.60     42.88  
min        0.08     10.00  
25%       14.37     65.00  
50%       26.06     83.00  
75%       43.43    108.00  
max      130.61    237.00  


In [ ]:
# Feature engineering means creating new columns from existing data
# that give the model additional signal to learn from.
# Raw pollutant readings alone are not enough — the model also needs
# to understand time patterns and recent trends.

# First make sure Datetime is in proper datetime format
# so we can extract hour, month etc from it
blr['Datetime'] = pd.to_datetime(blr['Datetime'])

# --- Time features ---
# Hour of day: pollution at 8am (rush hour) is very different from 3am
blr['hour'] = blr['Datetime'].dt.hour

# Month: captures seasonal patterns — winter months have worse AQI in India
blr['month'] = blr['Datetime'].dt.month

# Day of week: 0=Monday, 6=Sunday
# Weekday traffic and industrial activity differs from weekends
blr['day_of_week'] = blr['Datetime'].dt.dayofweek

# Is weekend: 1 if Saturday or Sunday, 0 otherwise
# Weekend AQI tends to be lower due to less traffic and industry
blr['is_weekend'] = (blr['day_of_week'] >= 5).astype(int)

# Is shift hour: 1 if the hour falls within a typical industrial work shift
# We define shift as 6am to 6pm — this is directly relevant to ShiftSafe AI
# Workers are exposed to pollution during these hours
blr['is_shift_hour'] = blr['hour'].between(6, 18).astype(int)

# --- Lag features ---
# Lag features tell the model what happened in recent past hours.
# AQI doesn't jump randomly — it builds and fades over time.
# "What was AQI 1 hour ago?" is one of the strongest predictors of now.

# AQI 1 hour ago — shift the AQI column forward by 1 row
blr['AQI_lag1'] = blr['AQI'].shift(1)

# AQI 3 hours ago — captures slightly longer trend
blr['AQI_lag3'] = blr['AQI'].shift(3)

# --- Rolling average features ---
# Rolling averages smooth out noise and capture pollution trends.
# "What was the average PM2.5 over the last 6 hours?"
# This tells the model whether pollution is building up or clearing.

# 6-hour rolling average of PM2.5
# min_periods=1 means it calculates even if fewer than 6 rows are available
# (important for the first few rows at the start of the dataset)
blr['PM25_rolling6'] = blr['PM2.5'].rolling(window=6, min_periods=1).mean()

# 6-hour rolling average of AQI
blr['AQI_rolling6'] = blr['AQI'].rolling(window=6, min_periods=1).mean()

# --- Drop rows with NaN created by lag ---
# The very first row has no "1 hour ago" value, so AQI_lag1 will be NaN.
# The first 3 rows will have NaN in AQI_lag3.
# We drop these few rows — losing 3 rows out of 48,192 is negligible.
blr = blr.dropna().reset_index(drop=True)

# Confirm all new columns are there and no NaN remains anywhere
print(f"Final shape: {blr.shape}")
print(f"\nAll columns: {blr.columns.tolist()}")
print(f"\nAny NaN remaining: {blr.isnull().sum().sum()}")
print(f"\nFirst row to verify new columns:")
print(blr[['Datetime','AQI','hour','month','is_shift_hour','AQI_lag1','PM25_rolling6']].head(3))

Final shape: (48189, 20)

All columns: ['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NH3', 'CO', 'SO2', 'O3', 'AQI', 'hour', 'month', 'day_of_week', 'is_weekend', 'is_shift_hour', 'AQI_lag1', 'AQI_lag3', 'PM25_rolling6', 'AQI_rolling6']

Any NaN remaining: 0

First row to verify new columns:
             Datetime    AQI  hour  month  is_shift_hour  AQI_lag1  \
0 2015-01-01 04:00:00   76.4     4      1              0      88.8   
1 2015-01-01 05:00:00  108.2     5      1              0      76.4   
2 2015-01-01 06:00:00   55.6     6      1              1     108.2   

   PM25_rolling6  
0      51.343000  
1      51.294000  
2      51.261333  


In [ ]:
# Save the fully cleaned and feature-engineered dataset to the processed folder.
# Every next step — XGBoost model, Bi-GRU, API — will load THIS file.
# Never modify the raw city_hour.csv — always work from blr_clean.csv going forward.

save_path = '/content/drive/MyDrive/ShiftSafe_AI/data/blr_clean.csv'
blr.to_csv(save_path, index=False)

# Confirm it saved correctly by reloading and checking shape
verify = pd.read_csv(save_path)
print(f"Saved successfully.")
print(f"Shape: {verify.shape}")
print(f"Columns: {verify.columns.tolist()}")
print(f"Missing values: {verify.isnull().sum().sum()}")

Saved successfully.
Shape: (48189, 20)
Columns: ['City', 'Datetime', 'PM2.5', 'PM10', 'NO', 'NO2', 'NH3', 'CO', 'SO2', 'O3', 'AQI', 'hour', 'month', 'day_of_week', 'is_weekend', 'is_shift_hour', 'AQI_lag1', 'AQI_lag3', 'PM25_rolling6', 'AQI_rolling6']
Missing values: 0
